In [0]:
catalog_name = "ct_oil_gas" 
bronze_table_name = f"oil_and_gas_data"
bronze_schema_name = "sc_bronze"
silver_schema_name = "sc_silver"


In [0]:
df = spark.table(f"{catalog_name}.{bronze_schema_name}.{bronze_table_name}")

df.limit(10).display()

In [0]:
from pyspark.sql import functions as F

duplicate_df = df.groupBy(df.transaction_id).count().filter(F.col("count") > 1)
duplicate_df.display()

In [0]:
duplicate_df.count()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql import DataFrame, SparkSession
from typing import List
from pyspark.sql import Row


class Transformation:
    """
    Applies configured transformations to a DataFrame.

    Behavior is driven by a transformation config (rows from a config
    table) rather than hardcoded column logic, so new columns/casts/
    cleanup rules can be added without code changes.
    """

    def __init__(self, df: DataFrame):
        """
        Args:
            df: The DataFrame to transform. Mutated in place across
                method calls via self.df.
        """
        self.df = df

    def drop_duplicates(self) -> DataFrame:
        """Drop fully duplicate rows across all columns."""
        self.df = self.df.dropDuplicates()
        return self.df

    def drop_nulls(self) -> DataFrame:
        """Drop rows containing any null values."""
        self.df = self.df.dropna()
        return self.df

    def deduplicate_by_key(self, key_column: str, order_by_column: str) -> DataFrame:
        """
        Keep only the first occurrence per key_column, ordered by order_by_column ascending.
        """

        w = Window.partitionBy(key_column).orderBy(order_by_column)
        self.df = (
            self.df.withColumn("row_num", F.row_number().over(w))
            .filter(F.col("row_num") == 1)
            .drop("row_num")
        )
        return self.df

    def apply_cast(self, column_name: str, target_type: str) -> DataFrame:
        """Cast a single column to the given Spark SQL type string."""
        self.df = self.df.withColumn(column_name, F.col(column_name).cast(target_type))
        return self.df

    def apply_trim(self, column_name: str) -> DataFrame:
        """Trim whitespace on a single string column."""
        self.df = self.df.withColumn(column_name, F.trim(F.col(column_name)))
        return self.df

    def apply_config(self, config_rows: List[Row]) -> DataFrame:
        """
        Apply a sequence of configured transformations to self.df
        """
        for row in config_rows:
            ttype = row["transformation_type"]
            params = row["transformation_params"] or {}

            if ttype == "cast":
                self.apply_cast(row["column_name"], row["target_type"])
            elif ttype == "trim":
                self.apply_trim(row["column_name"])
            elif ttype == "dedup_by_key":
                self.deduplicate_by_key(row["dedup_key"], row["dedup_order_by"])
            elif ttype == "drop_nulls":
                self.drop_nulls()
            elif ttype == "drop_duplicates":
                self.drop_duplicates()
            else:
                raise ValueError(f"Unknown transformation_type: {ttype}")

        return self.df